# Throughput Optimized Mode (High Throughput Serving) Guide

This guide explains Ray Serve's throughput optimized mode, which uses gRPC for replica-to-replica communication instead of Ray Core's actor call mechanism, significantly improving request throughput for high-volume workloads.

<div class="alert alert-block alert-info">
    <ol>
        <li>Overview</li>
        <li>When to Enable Throughput Optimized Mode</li>
        <li>How It Works</li>
        <li>Using the _by_reference API</li>
        <li>Configuration Options</li>
        <li>Serialization Options</li>
    </ol>
</div>

**imports**

In [ ]:
import time
from ray import serve
from ray.serve.handle import DeploymentHandle

## Overview

By default, Ray Serve uses Ray Core's actor call mechanism to route requests from one deployment to another (e.g., from an API gateway to a model deployment). This involves:

1. **Ray ObjectRefs**: Results are returned as `ObjectRef` objects that can be passed to other Ray tasks/actors
2. **Ray's distributed object store**: Data flows through Ray's plasma store
3. **Ray scheduling**: Ray's scheduler coordinates the request routing

While this provides powerful features like distributed object references and integration with Ray tasks, it introduces overhead that can limit throughput in high-volume serving scenarios.

**Throughput optimized mode** replaces Ray Core actor calls with direct **gRPC calls** between replicas, bypassing Ray's object store and scheduler for inter-deployment communication. This results in:

- **Higher throughput**: Up to 54% higher QPS, and up to 3x streaming tokens per second
- **Lower latency**: Reduced per-request overhead
- **More predictable performance**: Less variability from Ray scheduling

## When to Enable Throughput Optimized Mode


### Recommended Use Cases

Enable throughput optimized mode when:

1. **Latency-sensitive APIs**: Where every millisecond counts
2. **Model composition patterns**: Multiple deployments chained together (e.g., router → model → postprocessor)
3. **Small payloads**: Request/response data is under 4MB (the default gRPC message size limit)
4. **Can't benefit from zero-copy deserialization**: Replica requests go across nodes, ruling out any benefits from zero-copy deserialization

## How It Works

Let's go through the request flow in both modes.

### Architecture Comparison


#### Default Mode (Ray Core)

Below is a diagram of the default mode (Ray Core)

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_by_reference_true.png" alt="Ray Core Mode Diagram" style="width: 50%; height: auto;">


**Request Flow in Standard Mode:**

1. **Request arrives** at an ingress replica (via HTTP or gRPC proxy)
2. `DeploymentHandle.remote(obj)` called
2. The handle's router selects a **replica**
3. The handle will serialize the args and store them in **Ray Object Store** (if >100KB)
4. Ray actor call made with object references
5. Replica **fetches args from Object Store**
6. Replica executes request
7. The replica will serialize the response into **Object Store** (if >100KB)
8. The handle **fetches response from Object Store**
9. Response returned

Note: the 100KB limit is configurable by setting `RAY_max_direct_call_object_size`

#### Throughput Optimized Mode (gRPC)

Below is a diagram of the throughput optimized mode (gRPC)

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_by_reference_false.png" alt="Ray Core Mode Diagram" style="width: 50%; height: auto;">

**Request Flow in Throughput Optimized Mode:**
1. **Request arrives** at an ingress replica (via HTTP or gRPC proxy)
2. `DeploymentHandle.remote(obj)` called
3. The handle's router selects a **replica**
4. The handle will serialize the args and store them in gRPC message
5. gRPC call made to replica
6. Replica executes request
7. The replica will serialize the response into gRPC message
8. Response returned


## Using the `_by_reference` API

Below is an example of how to use the `_by_reference` API to leverage direct gRPC communication between replicas.

In [ ]:
@serve.deployment
class Model:
    def generate(self, data: dict) -> dict:
        time.sleep(1.0)  # Slow down to keep objects in store longer
        return {"output": f"Processed: {data.get('text', '')[:20].upper()}..."}


@serve.deployment
class Gateway:
    def __init__(self, model_handle: DeploymentHandle):
        # Configure handle for throughput optimized mode
        # Note: Add request_serialization="orjson" for faster JSON serialization
        # (requires `pip install orjson`)
        self._model = model_handle.options(
            _by_reference=False,
            # Optional: use faster serialization for JSON-compatible data
            # request_serialization="orjson",
            # response_serialization="orjson",
        )
    
    async def __call__(self, request):
        # This call goes via gRPC, not Ray Core
        result = await self._model.generate.remote(await request.json())
        return result

### Verifying Object Store Behavior

To verify the difference between `_by_reference=True` (default) and `_by_reference=False`, we'll use a script that monitors object store usage while sending concurrent requests.

In [ ]:
# Run with _by_reference=False (gRPC mode) - expect ~0 object store usage
!python scripts/by_reference_demo.py

Now let's run with `_by_reference=True` (Ray Core mode) to see non-zero object store usage:

In [ ]:
# Run with _by_reference=True (Ray Core mode) - expect non-zero object store usage
!python scripts/by_reference_demo.py --by-ref


### Note on Payload Size Limits

With `_by_reference=False` (gRPC mode), payloads exceeding **4MB** will fail due to gRPC message size limits. Test with a large payload:


In [ ]:
# This will fail with gRPC mode due to 4MB limit (5MB payload)
!python scripts/by_reference_demo.py --kb 5000 -n 5

With `_by_reference=True`, large payloads work because they go through the object store (no gRPC size limit):

In [ ]:
# This works with Ray Core mode - large payloads go through object store
!python scripts/by_reference_demo.py --by-ref --kb 5000 -n 5